## Sentiment Analysis for IMDB website - RNN 
- perform- sentiment Analysis for movie review  into positive & Negetive

##  RNN Sentiment Analysis Architecture

```
                +----------------------+
                |     Load Dataset     |
                | (Reviews, Labels)    |
                +----------+-----------+
                           |
                           v
                +----------------------+
                | Text Preprocessing   |
                | Lowercase            |
                | Remove Punctuation   |
                | Remove Urls, HTML    |
                | Stopwords(a,an,the)  |
                | Stemming             |
                | Tokenization         |
                | Vectoization (TF-IDF)|
                | Padding              |
                +----------+-----------+
                           |
                           v
                +----------------------+
                | Tensor Conversion    |
                | Dataset              |
                | DataLoader           |
                +----------+-----------+
                           |
                           v
                +----------------------+
                |      RNN Model       |
                | Embedding Layer      |
                | RNN Layer            |
                | Fully Connected      |
                | Sigmoid Output       |
                +----------+-----------+
                           |
               +-----------+-----------+
               |                       |
               v                       v
      +----------------+      +----------------+
      |     Train      |      |    Evaluate    |
      | Forward Pass   |      | Accuracy       |
      | Loss           |      | Precision      |
      | Backpropagation|      | Recall, F1     |
      +--------+-------+      +----------------+
               |
               v
      +------------------------+
      |  New Review Prediction |
      +-----------+------------+
                  |
          +-------+-------+
          |               |
          v               v
     Positive (1)    Negative (0)
```


In [28]:
import pandas as pd
import re

import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from nltk.stem import PorterStemmer
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import TensorDataset, DataLoader

import torch.nn as nn
import torch.optim as optim


df = pd.read_csv("IMDB Dataset.csv")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


In [13]:
df.shape

(50000, 2)

In [14]:
df.head(20)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
5,"Probably my all-time favorite movie, a story o...",positive
6,I sure would like to see a resurrection of a u...,positive
7,"This show was an amazing, fresh & innovative i...",negative
8,Encouraged by the positive comments about this...,negative
9,If you like original gut wrenching laughter yo...,positive


In [15]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

- NO Null value present 

In [21]:
df.drop_duplicates(inplace = True)
df.shape

(49582, 2)

## Pre-processing -------------------------------------------------

In [23]:
### 1. Converting to lowercase

df["review"] = df["review"].str.lower()

In [ ]:
def remove_urls(text):
    text = re.sub(r"http\S+" , "", text)   
    return text
    
df["review"] = df["review"].apply(remove_urls)

In [ ]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]" , "", text) 
    return text

df["review"] = df["review"].apply(remove_punctuations)

In [ ]:
df.head()

In [ ]:
def remove_html(text):
    text = re.sub(r"<.*?>" , "", text)
    return text

df["review"] = df["review"].apply(remove_html)

In [ ]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")

    return text

df["review"] = df["review"].apply(remove_stopwords)

In [ ]:
df.head()

In [ ]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [ ]:
le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [ ]:
y = df["sentiment"]

In [ ]:
df.head()

In [ ]:
tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [ ]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [ ]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

In [ ]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        out, _ = self.rnn(x, h0) 
        

        out = self.fc(out[:, -1, :])
        return out

In [ ]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

In [ ]:
## Training the RNN
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction
        
        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

In [ ]:
# evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")